# L02 — Delta Lake Deep Dive: Schema, Constraints, Table Types, Clones, Maintenance

Builds on Lesson 1 (DML, history, time travel, RESTORE, UNDROP). This lesson covers the Delta transaction log, schema enforcement and evolution, constraints, generated/identity columns, managed vs external tables, clones, and table maintenance — skills tested on the Data Engineer Associate and Professional exams.

### Learning objectives
1. **Use** schema enforcement, schema evolution, and constraints to protect data quality.
2. **Explain** how the Delta transaction log works and how a read resolves the current version.
3. **Compare** managed vs external tables, shallow vs deep clones, OPTIMIZE vs VACUUM.
4. **Implement** generated columns, identity columns, MERGE WITH SCHEMA EVOLUTION, CLONE, and maintenance operations.
5. **Diagnose** write-time rejections, constraint violations, and the effect of predictive optimization on history.

> **Evidence-only rule:** every number and error in this notebook comes from a cell that actually ran on this serverless compute, or from official Databricks docs (named inline). If a behaviour could not be verified here it is marked **NOT VERIFIED ON SERVERLESS — concept only**.

## 2 Warm-Up (Lesson 1 recall — no hints)

Type these from memory. Each task has a validation cell right after it. Solutions are in section 11 — do not peek.

> The warm-up uses its own tiny tables, so it runs before the classroom setup.

### W1 — MERGE with all three branches

Create a tiny target `(id INT, status STRING)` with rows `(1,'old'),(2,'old'),(3,'old')` and a source `(id INT, status STRING)` with rows `(1,'new'),(4,'new')`. MERGE so that: matched rows UPDATE, unmatched source rows INSERT, unmatched target rows DELETE. Expected result: 2 rows `(1,'new'),(4,'new')`.

Use the schema `main.dbacademy_l02` and table names `wu1_target` / `wu1_source`.

In [0]:
%python
# W1: create target + source, then MERGE with all three branches.
# TODO: your code here


In [0]:
%python
# W1 validation
try:
    rows = spark.sql("SELECT id, status FROM main.dbacademy_l02.wu1_target ORDER BY id").collect()
    if len(rows) == 2 and rows[0]["id"] == 1 and rows[0]["status"] == "new" and rows[1]["id"] == 4 and rows[1]["status"] == "new":
        print("✅ Passed — W1 MERGE (2 rows: (1,new),(4,new))")
    else:
        print(f"Hint: expected [(1,new),(4,new)], got {[(r['id'],r['status']) for r in rows]}")
except Exception as e:
    print("Hint: write and run the W1 code cell first.", str(e).splitlines()[0])

### W2 — Time travel to the first version

Create table `wu2_t` in `main.dbacademy_l02` via `CREATE TABLE ... AS SELECT` with 5 rows (so version 0 already has 5 rows), then `INSERT` 5 more rows. Query `VERSION AS OF 0` and compare its count to the current count. Expected: version 0 = 5, current = 10.

In [0]:
%python
# W2: create a table with two inserts, then compare VERSION AS OF 0 to current.
# TODO: your code here


In [0]:
%python
# W2 validation
try:
    v0 = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l02.wu2_t VERSION AS OF 0").collect()[0]["c"]
    cur = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l02.wu2_t").collect()[0]["c"]
    if v0 == 5 and cur == 10:
        print(f"✅ Passed — W2 time travel (v0={v0}, current={cur})")
    else:
        print(f"Hint: v0(={v0} want 5) current(={cur} want 10)")
except Exception as e:
    print("Hint: write and run the W2 code cell first.", str(e).splitlines()[0])

### W3 — RESTORE and show the new RESTORE version

Create table `wu3_t (id INT)` in `main.dbacademy_l02`, insert 10 rows, delete 4 (leaving 6), then `RESTORE TABLE ... TO VERSION AS OF` the version *before* the delete. Show that history now contains a `RESTORE` operation and the row count is back to 10.

> **Serverless tip:** RESTORE can race with predictive optimization. Wrap it in a retry loop (try/except, sleep, retry a few times) so a `ConcurrentWriteException` does not fail the cell.

In [0]:
%python
# W3: create, insert 10, delete 4, RESTORE to the pre-delete version, show history.
# TODO: your code here (use a retry loop around RESTORE)


In [0]:
%python
# W3 validation
try:
    cnt = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l02.wu3_t").collect()[0]["c"]
    ops = [r["operation"] for r in spark.sql("DESCRIBE HISTORY main.dbacademy_l02.wu3_t").select("operation").collect()]
    has_restore = "RESTORE" in ops
    if has_restore and cnt == 10:
        print(f"✅ Passed — W3 RESTORE (count={cnt}, RESTORE in history)")
    else:
        print(f"Hint: count(={cnt} want 10) RESTORE in history={has_restore}")
except Exception as e:
    print("Hint: write and run the W3 code cell first.", str(e).splitlines()[0])

In [0]:
%python
# Lesson 2 — Classroom Setup (idempotent)
# Schema: main.dbacademy_l02.  Volume: /Volumes/main/dbacademy_l02/data
# Isolated RNG so all counts are deterministic across runs.
import random, datetime
from pyspark.sql import Row

rng = random.Random(42)  # isolated: Spark calls cannot consume this generator's state

spark.sql("USE CATALOG main")
spark.sql("DROP SCHEMA IF EXISTS main.dbacademy_l02 CASCADE")
spark.sql("CREATE SCHEMA IF NOT EXISTS main.dbacademy_l02")
spark.sql("CREATE VOLUME IF NOT EXISTS main.dbacademy_l02.data")

# --- name pools for fake (no PHI) healthcare data ---
first_names = ["James","Mary","John","Patricia","Robert","Jennifer","Michael","Linda",
               "William","Elizabeth","David","Barbara","Richard","Susan","Joseph","Jessica"]
last_names  = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis",
               "Rodriguez","Martinez","Hernandez","Lopez","Gonzalez","Wilson","Anderson","Thomas"]
genders = ["F","M"]
enc_types = ["OUTPATIENT","INPATIENT","EMERGENCY","TELEHEALTH"]
diag_codes = ["Z00.00","E11.9","I10","J06.9","M54.5","R51.9","K21.9","N39.0"]
lab_tests = ["GLUCOSE","HEMOGLOBIN","CHOLESTEROL","SODIUM","POTASSIUM","CREATININE","WBC","PLATELETS"]
base_date = datetime.date(2024, 1, 1)

# --- generate ALL rows in pure Python (no Spark calls here) ---
patients = [Row(patient_id=i,
                 patient_name=f"{rng.choice(first_names)} {rng.choice(last_names)}",
                 dob=(base_date - datetime.timedelta(days=rng.randint(7000, 30000))).isoformat(),
                 gender=rng.choice(genders),
                 city=rng.choice(["Boston","Springfield","Hartford","Providence","Portland"]),
                 created_at=(base_date + datetime.timedelta(days=rng.randint(0, 400))).isoformat())
           for i in range(1, 51)]

encounters = [Row(encounter_id=i, patient_id=rng.randint(1, 50), provider_id=rng.randint(1, 10),
                  encounter_date=(base_date + datetime.timedelta(days=rng.randint(0, 365))).isoformat(),
                  encounter_type=rng.choice(enc_types), diagnosis_code=rng.choice(diag_codes))
           for i in range(1, 301)]

# lab_results: 500 rows; a few NULL patient_ids and some negative result_values
lab_results = []
for i in range(1, 501):
    pid = None if i % 37 == 0 else rng.randint(1, 50)          # ~13 NULL patient_ids
    val = round(rng.uniform(50, 200), 2)
    if i % 19 == 0:
        val = round(rng.uniform(-10, -1), 2)                    # ~26 negative values
    lab_results.append(Row(lab_id=i, patient_id=pid,
                           test_name=rng.choice(lab_tests),
                           result_value=val,
                           result_date=(base_date + datetime.timedelta(days=rng.randint(0, 365))).isoformat()))

# --- write all tables (after all random generation is done) ---
spark.createDataFrame(patients).write.mode("overwrite").saveAsTable("main.dbacademy_l02.patients")
spark.createDataFrame(encounters).write.mode("overwrite").saveAsTable("main.dbacademy_l02.encounters")
spark.createDataFrame(lab_results).write.mode("overwrite").saveAsTable("main.dbacademy_l02.lab_results")

# --- verify real counts with asserts ---
expected = {"patients": 50, "encounters": 300, "lab_results": 500}
for t, n in expected.items():
    cnt = spark.table(f"main.dbacademy_l02.{t}").count()
    assert cnt == n, f"{t}: expected {n}, got {cnt}"
    print(f"{t}: {cnt}")
null_pids = spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.lab_results WHERE patient_id IS NULL").collect()[0][0]
neg_vals  = spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.lab_results WHERE result_value < 0").collect()[0][0]
print(f"lab_results with NULL patient_id: {null_pids}")
print(f"lab_results with negative result_value: {neg_vals}")
assert null_pids > 0 and neg_vals > 0, "need NULLs and negatives for constraint demos"
print("SETUP OK — schema main.dbacademy_l02 ready")

patients: 50
encounters: 300
lab_results: 500
lab_results with NULL patient_id: 13
lab_results with negative result_value: 26
SETUP OK — schema main.dbacademy_l02 ready


## 4 Concepts

### 4.1 The Delta transaction log on disk

**What:** Every Delta table stores a transaction log in a `_delta_log` folder. Each commit is a JSON file named `00000000000000000000.json`, `00000000000000000001.json`, … Each JSON commit records the actions of that operation: `add` (new data file), `remove` (file no longer active), `metaData`, `protocol`, `commitInfo`, etc. Periodically a **checkpoint** (Parquet file, e.g. `00000000000000000010.checkpoint.parquet`) is written — by default every 10 commits (`delta.checkpointInterval`).

**Why:** The log is the source of truth. ACID guarantees, time travel, and schema enforcement all come from replaying the log — not from the data files.

**How a read resolves the current version:** Spark finds the latest checkpoint, replays only the JSON commits *after* it, and applies the `add`/`remove` actions to compute the set of **active** data files for the latest version. To read an older version it replays the log up to that version number.

| Tool | Shows |
| --- | --- |
| `DESCRIBE HISTORY t` | commit-level audit trail (version, operation, timestamp, metrics) |
| `DESCRIBE DETAIL t` | physical info (format, location, numFiles, sizeInBytes, properties) |
| `DESCRIBE EXTENDED t` | schema + extended metadata (Type, Provider, Location) |

### 4.2 Schema enforcement vs schema evolution

**Schema enforcement (on by default):** Delta rejects any write whose schema does not match the table's — an extra column, a missing column, or a type mismatch all fail at write time.

**Schema evolution** lets the schema change to accept new data:

| Option | Mode | Effect |
| --- | --- | --- |
| `.option("mergeSchema","true")` | append / overwrite | merges NEW columns into the schema; old rows get `NULL` in new columns |
| `.option("overwriteSchema","true")` | overwrite only | replaces the entire schema (drops columns not in the new write) |
| `spark.databricks.delta.schema.autoMerge.enabled=true` | session | enables mergeSchema by default for all writes |

**DDL schema changes:**

| Statement | Notes |
| --- | --- |
| `ALTER TABLE t ADD COLUMN (col TYPE)` | adds a column |
| `ALTER TABLE t RENAME COLUMN old TO new` | needs `delta.columnMapping.mode = 'name'` |
| `ALTER TABLE t DROP COLUMN col` | needs `delta.columnMapping.mode = 'name'` |
| `MERGE WITH SCHEMA EVOLUTION INTO t ...` | lets a MERGE evolve the target schema |
| `ALTER TABLE t ALTER COLUMN col TYPE bigint` | needs `delta.enableTypeWidening = 'true'` (DBR 15.4+) |

> **Exam point:** `mergeSchema` adds columns; `overwriteSchema` replaces the schema. RENAME/DROP COLUMN require column mapping mode `name`.

### 4.3 Constraints

**What:** Delta enforces two kinds of constraints at **write time**:

| Constraint | Syntax | When it fires |
| --- | --- | --- |
| `NOT NULL` | `col TYPE NOT NULL` or `ALTER TABLE t ALTER COLUMN col SET NOT NULL` | any INSERT/UPDATE that would leave the column NULL |
| `CHECK` | `ALTER TABLE t ADD CONSTRAINT chk CHECK (condition)` | any INSERT/UPDATE where the condition is false |

**Existing rows:** when you ADD a constraint, Delta validates all existing rows first. If any row violates it, the ALTER **fails** — you must clean the bad rows before adding the constraint.

**Primary / foreign keys in Unity Catalog:** per the [Constraints docs](https://docs.databricks.com/aws/en/tables/constraints/), PK/FK/UNIQUE constraints are **informational only and are not enforced**. They may help the query optimizer but never reject a bad write. Only NOT NULL and CHECK are enforced.

**Show constraints:** `SHOW TBLPROPERTIES t` and `DESCRIBE EXTENDED t` list the constraints.

### 4.4 Generated columns, identity columns, DEFAULT values

**Generated column** — computed from other columns, never stored as input:
```sql
encounter_year INT GENERATED ALWAYS AS (YEAR(encounter_date))
```
You cannot insert an explicit value into a `GENERATED ALWAYS` column.

**Identity column** — auto-incrementing surrogate key:
```sql
row_id BIGINT GENERATED ALWAYS AS IDENTITY        -- cannot insert explicit values
row_id BIGINT GENERATED BY DEFAULT AS IDENTITY    -- CAN insert explicit values
```

**DEFAULT values:**
```sql
status STRING DEFAULT 'PENDING'
```

**Limits (per docs):** identity columns cannot be partition columns; generated columns cannot call non-deterministic functions. See the [Generated columns](https://docs.databricks.com/aws/en/delta/generated-columns/) and [Identity columns](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-identity-columns/) docs.

### 4.5 Managed vs external tables in Unity Catalog

| | Managed | External |
| --- | --- | --- |
| Data location | UC-managed (under the catalog's storage) | You specify `LOCATION` |
| `DESCRIBE EXTENDED` "Type" | `MANAGED` | `EXTERNAL` |
| `DROP TABLE` | deletes metadata **and** data files | deletes metadata **only**; data files remain |
| `UNDROP TABLE` | recovers the table within the retention window | recovers metadata; data must still exist at LOCATION |

**External tables** need a UC external location or a UC volume path. On this serverless workspace we test what we can and record the real result (E8).

> **Exam point:** `DROP` on a managed table deletes the data; `DROP` on an external table does not. `UNDROP` can recover a managed table (verified in Lesson 1 / E8).

### 4.6 CLONE — shallow vs deep

| | SHALLOW CLONE | DEEP CLONE |
| --- | --- | --- |
| What it copies | metadata only — shares the source's data files (copy-on-write) | metadata **and** data files |
| Cost | very cheap (no data copy) | expensive (copies all data) |
| Write to clone | creates new files owned by the clone; source is unchanged | same — writes are independent |
| Use case | dev/test copy, quick backup | independent copy for transformations |

```sql
CREATE TABLE dev_encounters SHALLOW CLONE main.dbacademy_l02.encounters;
CREATE TABLE backup_encounters DEEP CLONE main.dbacademy_l02.encounters;
```

> **Exam point:** a shallow clone does NOT copy data files — it points at the source's files. Writes to the clone do not affect the source (E9).

### 4.7 Table maintenance

| Operation | What it does | Key detail |
| --- | --- | --- |
| `OPTIMIZE` | compacts small files into larger ones | `operationMetrics`: `numRemovedFiles`, `numAddedFiles` |
| `OPTIMIZE ... ZORDER BY (col)` | co-locates rows by column value | legacy; **liquid clustering** is the modern default (later lesson) |
| `VACUUM` | removes data files no longer referenced by the log | must respect `delta.deletedFileRetentionDuration` (default 7 days) |
| `VACUUM ... DRY RUN` | lists files that would be removed | no data is deleted |
| `REORG TABLE ... APPLY (PURGE)` | rewrites files to physically remove rows marked by deletion vectors | purges soft-deleted data |

**Retention properties:**

| Property | Default | Controls |
| --- | --- | --- |
| `delta.deletedFileRetentionDuration` | 7 days | how long removed files are kept before VACUUM can delete them |
| `delta.logRetentionDuration` | 30 days | how long the transaction log is kept |

> **VACUUM and time travel:** VACUUM removes old data files, which limits how far back you can time-travel. `spark.databricks.delta.retentionDurationCheck.enabled` (default `true`) blocks a VACUUM with retention below 7 days — we test this in E11.
>
> **Predictive optimization** (ON here) runs OPTIMIZE/VACUUM automatically and adds `OPTIMIZE` versions to history you did not write. Read the `operation` column.

## 5 Examples

Each example has a **Predict** cell (guess before running), a **code** cell, and an **Explained** cell with the real numbers from this run.

### Example E1 — The Delta transaction log on disk

**Predict:**
1. After `CREATE TABLE` + 3 `INSERT` statements, how many versions are in `DESCRIBE HISTORY`?
2. Can you list the `_delta_log` JSON files with `dbutils.fs.ls` on a UC managed table on serverless?
3. What does `DESCRIBE DETAIL` show for `numFiles` and `format`?

In [0]:
%python
# E1: inspect the Delta transaction log of a managed table
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex1_log")
spark.sql("CREATE TABLE main.dbacademy_l02.ex1_log (id INT, val STRING)")
spark.sql("INSERT INTO main.dbacademy_l02.ex1_log VALUES (1,'a')")   # write 1
spark.sql("INSERT INTO main.dbacademy_l02.ex1_log VALUES (2,'b')")   # write 2
spark.sql("INSERT INTO main.dbacademy_l02.ex1_log VALUES (3,'c')")   # write 3
detail = spark.sql("DESCRIBE DETAIL main.dbacademy_l02.ex1_log").collect()[0]
print("format:", detail["format"], "| numFiles:", detail["numFiles"], "| sizeInBytes:", detail["sizeInBytes"])
print("location (managed, often empty on serverless):", repr(detail["location"]))
# Try to list the _delta_log directory
print("dbutils.fs.ls _delta_log (expect to fail on serverless):")
try:
    dfiles = dbutils.fs.ls(detail["location"] + "/_delta_log/")
    print("  files:", [f.name for f in dfiles][:5])
except Exception as e:
    print("  failed:", str(e).splitlines()[0])
display(spark.sql("DESCRIBE HISTORY main.dbacademy_l02.ex1_log").select("version","operation").orderBy("version"))

format: delta | numFiles: 3 | sizeInBytes: 3006
location (managed, often empty on serverless): ''
dbutils.fs.ls _delta_log (expect to fail on serverless):
  failed: [DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /_delta_log SQLSTATE: 56038


version,operation
0,CREATE TABLE
1,WRITE
2,WRITE
3,WRITE


**Explained (real output from this run):**

* `DESCRIBE DETAIL`: `format=delta`, `numFiles=3` (one file per INSERT), `sizeInBytes=3006`. The `location` was an **empty string** — the managed storage path is not exposed on this serverless workspace.
* `dbutils.fs.ls` on the `_delta_log` directory **failed** with `[DBFS_DISABLED] Public DBFS root is disabled`. On this compute you cannot list the raw transaction-log files of a UC managed table.
* `DESCRIBE HISTORY` showed **4 versions**: v0 `CREATE TABLE`, v1–v3 `WRITE` — one commit per write.

> **NOT VERIFIED ON SERVERLESS — concept only:** the JSON commit files (`000…json`) and checkpoint Parquet files inside `_delta_log` could not be listed here because of `DBFS_DISABLED`. Their structure (`add`/`remove`/`metaData`/`commitInfo` actions, checkpoint every 10 commits) is described in concept 4.1 and the [Delta transaction log docs](https://docs.databricks.com/aws/en/delta/delta-intro/).

### Example E2 — Schema enforcement and mergeSchema

**Predict:**
1. You append a DataFrame with an **extra column** to a Delta table. What happens?
2. Same append with `.option("mergeSchema","true")` — does it succeed?
3. After a successful mergeSchema append, what value do the **old** rows have in the new column?

In [0]:
%python
# E2: schema enforcement rejects an extra column; mergeSchema accepts it
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex2_schema")
spark.sql("CREATE TABLE main.dbacademy_l02.ex2_schema (id INT, val STRING)")
spark.sql("INSERT INTO main.dbacademy_l02.ex2_schema VALUES (1,'a'),(2,'b')")
df_extra = spark.createDataFrame([(3,"c","extra")], "id INT, val STRING, note STRING")
print("append with extra column (expect fail):")
try:
    df_extra.write.mode("append").saveAsTable("main.dbacademy_l02.ex2_schema")
    print("  unexpectedly succeeded")
except Exception as e:
    print("  failed:", str(e).splitlines()[0])
print("append with mergeSchema=true (expect succeed):")
try:
    df_extra.write.mode("append").option("mergeSchema","true").saveAsTable("main.dbacademy_l02.ex2_schema")
    print("  succeeded")
except Exception as e:
    print("  failed:", str(e).splitlines()[0])
display(spark.sql("SELECT * FROM main.dbacademy_l02.ex2_schema ORDER BY id"))
print("old rows with NULL note:", spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.ex2_schema WHERE note IS NULL").collect()[0][0])

append with extra column (expect fail):
  failed: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
append with mergeSchema=true (expect succeed):
  succeeded


id,val,note
1,a,null
2,b,null
3,c,extra


old rows with NULL note: 2


**Explained (real output from this run):**

* Append with an extra column **failed**: `[DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.`
* Same append with `.option("mergeSchema","true")` **succeeded**.
* The table now has 3 rows: `(1,'a',NULL)`, `(2,'b',NULL)`, `(3,'c','extra')`. The **2 old rows have `NULL`** in the new `note` column.

> **Exam point:** schema enforcement rejects extra columns. `mergeSchema` merges the new column; existing rows get `NULL`. `overwriteSchema` (overwrite mode only) replaces the whole schema instead.

### Example E3 — Type mismatch and type widening

**Predict:**
1. You append a DataFrame whose `id` column is `STRING` (value `'x'`) into an `INT` column. What happens?
2. Can you permanently widen an `INT` column to `BIGINT`? What property must you enable first?

In [0]:
%python
# E3: a value that cannot be cast fails at write time; type widening changes the column type
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex3_type")
spark.sql("CREATE TABLE main.dbacademy_l02.ex3_type (id INT, val STRING)")
spark.sql("INSERT INTO main.dbacademy_l02.ex3_type VALUES (1,'a')")
df_str = spark.createDataFrame([("x","b")], "id STRING, val STRING")
print("append STRING 'x' into INT column (expect fail):")
try:
    df_str.write.mode("append").saveAsTable("main.dbacademy_l02.ex3_type")
    print("  unexpectedly succeeded")
except Exception as e:
    print("  failed:", str(e).splitlines()[0])
# type widening: enable the feature, then ALTER COLUMN TYPE
print("enable type widening + ALTER COLUMN id TYPE BIGINT:")
try:
    spark.sql("ALTER TABLE main.dbacademy_l02.ex3_type SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true')")
    spark.sql("ALTER TABLE main.dbacademy_l02.ex3_type ALTER COLUMN id TYPE BIGINT")
    print("  succeeded")
except Exception as e:
    print("  failed:", str(e).splitlines()[0])
display(spark.sql("DESCRIBE main.dbacademy_l02.ex3_type").select("col_name","data_type"))

append STRING 'x' into INT column (expect fail):
  failed: [CAST_INVALID_INPUT] The value 'x' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
enable type widening + ALTER COLUMN id TYPE BIGINT:
  succeeded


col_name,data_type
id,bigint
val,string


**Explained (real output from this run):**

* Appending `STRING 'x'` into an `INT` column **failed** with `[CAST_INVALID_INPUT] The value 'x' of the type "STRING" cannot be cast to "INT" because it is malformed.` (Note: numeric types like `BIGINT`→`INT` are auto-cast by Spark; only a value that truly cannot be cast is rejected.)
* Type widening **succeeded**: after `ALTER TABLE ... SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true')`, `ALTER COLUMN id TYPE BIGINT` changed the column to `bigint` without rewriting data. (Per the [type widening docs](https://docs.databricks.com/aws/en/tables/features/type-widening/), this requires DBR 15.4 LTS+.)

> **Exam point:** `ALTER COLUMN ... TYPE` to a *wider* type requires `delta.enableTypeWidening = true`. Without it the ALTER fails.

### Example E4 — ADD / RENAME / DROP COLUMN and column mapping

**Predict:**
1. `ALTER TABLE ADD COLUMN` — does it work without any special property?
2. `ALTER TABLE RENAME COLUMN` — does it work without column mapping?
3. After enabling `delta.columnMapping.mode = 'name'`, do RENAME and DROP succeed?

In [0]:
%python
# E4: ADD COLUMN works; RENAME/DROP need column mapping mode 'name'
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex4_cols")
spark.sql("CREATE TABLE main.dbacademy_l02.ex4_cols (id INT, val STRING)")
spark.sql("INSERT INTO main.dbacademy_l02.ex4_cols VALUES (1,'a')")
spark.sql("ALTER TABLE main.dbacademy_l02.ex4_cols ADD COLUMN (note STRING)")   # ADD works
print("ADD COLUMN succeeded")
print("RENAME COLUMN without column mapping (expect fail):")
try:
    spark.sql("ALTER TABLE main.dbacademy_l02.ex4_cols RENAME COLUMN val TO value")
    print("  unexpectedly succeeded")
except Exception as e:
    print("  failed:", str(e).splitlines()[0])
# enable column mapping, then RENAME and DROP
spark.sql("ALTER TABLE main.dbacademy_l02.ex4_cols SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')")
print("column mapping enabled")
try:
    spark.sql("ALTER TABLE main.dbacademy_l02.ex4_cols RENAME COLUMN val TO value")
    print("  RENAME succeeded")
except Exception as e:
    print("  RENAME failed:", str(e).splitlines()[0])
try:
    spark.sql("ALTER TABLE main.dbacademy_l02.ex4_cols DROP COLUMN note")
    print("  DROP COLUMN succeeded")
except Exception as e:
    print("  DROP failed:", str(e).splitlines()[0])
display(spark.sql("DESCRIBE main.dbacademy_l02.ex4_cols").select("col_name","data_type"))

ADD COLUMN succeeded
RENAME COLUMN without column mapping (expect fail):
  failed: [DELTA_UNSUPPORTED_RENAME_COLUMN.ENABLE_COLUMN_MAPPING] Column rename is not supported for your Delta table. Enable Column Mapping on your Delta table with mapping mode 'name' by running:
column mapping enabled
  RENAME succeeded
  DROP COLUMN succeeded


col_name,data_type
id,int
value,string


**Explained (real output from this run):**

* `ADD COLUMN` **succeeded** without any special property.
* `RENAME COLUMN` **failed** with `[DELTA_UNSUPPORTED_RENAME_COLUMN.ENABLE_COLUMN_MAPPING] Column rename is not supported for your Delta table. Enable Column Mapping ...`.
* After `SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')`, both `RENAME COLUMN val TO value` and `DROP COLUMN note` **succeeded**.
* Final schema: `id INT, value STRING` (the `note` column was dropped, `val` was renamed to `value`).

> **Exam point:** RENAME and DROP COLUMN require `delta.columnMapping.mode = 'name'`. ADD COLUMN does not.

### Example E5 — CHECK and NOT NULL constraints (enforced)

**Predict:**
1. You `ADD CONSTRAINT chk CHECK (result_value >= 0)` on `lab_results` which has **26 rows** with negative `result_value`. Does the ADD succeed?
2. After cleaning the negative rows and adding the CHECK constraint, what happens if you `INSERT` a row with `result_value = -5.0`?
3. You `ALTER COLUMN patient_id SET NOT NULL` on a table with **13 NULL patient_ids**. Does the ALTER succeed?

In [0]:
%python
# E5: CHECK and NOT NULL are enforced at write time
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex5_lr")
spark.sql("CREATE TABLE main.dbacademy_l02.ex5_lr AS SELECT * FROM main.dbacademy_l02.lab_results")
neg = spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.ex5_lr WHERE result_value < 0").collect()[0][0]
print(f"negative rows before: {neg}")
print("add CHECK (result_value >= 0) with negatives (expect fail):")
try:
    spark.sql("ALTER TABLE main.dbacademy_l02.ex5_lr ADD CONSTRAINT chk_nonneg CHECK (result_value >= 0)")
    print("  unexpectedly succeeded")
except Exception as e:
    print("  failed:", str(e).splitlines()[0])
# clean the bad rows, then add the constraint
spark.sql("DELETE FROM main.dbacademy_l02.ex5_lr WHERE result_value < 0")
print(f"negative rows after delete: {spark.sql('SELECT COUNT(*) FROM main.dbacademy_l02.ex5_lr WHERE result_value < 0').collect()[0][0]}")
spark.sql("ALTER TABLE main.dbacademy_l02.ex5_lr ADD CONSTRAINT chk_nonneg CHECK (result_value >= 0)")
print("CHECK constraint added after cleaning")
print("insert violating row (expect fail):")
try:
    spark.sql("INSERT INTO main.dbacademy_l02.ex5_lr (lab_id, patient_id, test_name, result_value, result_date) VALUES (99999, 1, 'GLUCOSE', -5.0, '2024-01-01')")
    print("  unexpectedly succeeded")
except Exception as e:
    print("  failed:", str(e).splitlines()[0])
# NOT NULL on patient_id where NULLs exist -> fail
print("ALTER COLUMN patient_id SET NOT NULL with NULLs (expect fail):")
try:
    spark.sql("ALTER TABLE main.dbacademy_l02.ex5_lr ALTER COLUMN patient_id SET NOT NULL")
    print("  unexpectedly succeeded")
except Exception as e:
    print("  failed:", str(e).splitlines()[0])
# show the constraint in TBLPROPERTIES
props = spark.sql("SHOW TBLPROPERTIES main.dbacademy_l02.ex5_lr").collect()
for r in props:
    if "constraint" in r["key"].lower() or "chk" in r["key"].lower():
        print(f"TBLPROPERTIES: {r['key']} = {r['value']}")

negative rows before: 26
add CHECK (result_value >= 0) with negatives (expect fail):
  failed: [DELTA_NEW_CHECK_CONSTRAINT_VIOLATION] 26 rows in main.dbacademy_l02.ex5_lr violate the new CHECK constraint (result_value >= 0).
negative rows after delete: 0
CHECK constraint added after cleaning
insert violating row (expect fail):
  failed: (com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException) [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint chk_nonneg (result_value >= 0) violated by row with values:
ALTER COLUMN patient_id SET NOT NULL with NULLs (expect fail):
  failed: [DELTA_NEW_NOT_NULL_VIOLATION] 13 rows in main.dbacademy_l02.ex5_lr violate the new NOT NULL constraint on patient_id.
TBLPROPERTIES: delta.constraints.chk_nonneg = result_value >= 0
TBLPROPERTIES: delta.feature.checkConstraints = supported


**Explained (real output from this run):**

* Adding `CHECK (result_value >= 0)` on the table with **26 negative rows** **failed** with `[DELTA_NEW_CHECK_CONSTRAINT_VIOLATION] 26 rows ... violate the new CHECK constraint (result_value >= 0).`
* After deleting the 26 negative rows, the ADD succeeded.
* `INSERT` of `result_value = -5.0` **failed** with `[DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint chk_nonneg (result_value >= 0) violated ...`
* `ALTER COLUMN patient_id SET NOT NULL` with **13 NULL patient_ids** **failed** with `[DELTA_NEW_NOT_NULL_VIOLATION] 13 rows ... violate the new NOT NULL constraint on patient_id.`
* `SHOW TBLPROPERTIES` showed: `delta.constraints.chk_nonneg = result_value >= 0` and `delta.feature.checkConstraints = supported`.

> **Exam point:** CHECK and NOT NULL are the only **enforced** constraints. Adding either validates existing rows first — if any row violates it, the ALTER fails. PK/FK/UNIQUE in UC are informational only (concept 4.3).

### Example E6 — Generated columns, identity columns, DEFAULT values

**Predict:**
1. You create a table with `row_id BIGINT GENERATED ALWAYS AS IDENTITY` and `encounter_year INT GENERATED ALWAYS AS (YEAR(encounter_date))`. You insert `(encounter_id, encounter_date)` only. What values do `row_id`, `encounter_year`, and `status` (with `DEFAULT 'PENDING'`) get?
2. Can you insert an explicit value into a `GENERATED ALWAYS AS IDENTITY` column?

In [0]:
%python
# E6: generated column, identity column, DEFAULT value
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex6_gen")
spark.sql("""
CREATE TABLE main.dbacademy_l02.ex6_gen (
  row_id BIGINT GENERATED ALWAYS AS IDENTITY,
  encounter_id INT,
  encounter_date DATE,
  encounter_year INT GENERATED ALWAYS AS (YEAR(encounter_date)),
  status STRING DEFAULT 'PENDING'
) TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
""")
spark.sql("INSERT INTO main.dbacademy_l02.ex6_gen (encounter_id, encounter_date) VALUES (1, '2024-06-15')")
display(spark.sql("SELECT * FROM main.dbacademy_l02.ex6_gen"))
print("explicit insert into GENERATED ALWAYS AS IDENTITY (expect fail):")
try:
    spark.sql("INSERT INTO main.dbacademy_l02.ex6_gen (row_id, encounter_id, encounter_date) VALUES (100, 2, '2024-07-01')")
    print("  unexpectedly succeeded")
except Exception as e:
    print("  failed:", str(e).splitlines()[0])

row_id,encounter_id,encounter_date,encounter_year,status
1,1,2024-06-15,2024,PENDING


explicit insert into GENERATED ALWAYS AS IDENTITY (expect fail):
  failed: [DELTA_IDENTITY_COLUMNS_EXPLICIT_INSERT_NOT_SUPPORTED] Providing values for GENERATED ALWAYS AS IDENTITY column row_id is not supported.


**Explained (real output from this run):**

* The single row after insert was: `row_id=1`, `encounter_id=1`, `encounter_date=2024-06-15`, `encounter_year=2024`, `status=PENDING`.
  * `row_id` was auto-generated (IDENTITY starts at 1).
  * `encounter_year` was computed as `YEAR('2024-06-15') = 2024`.
  * `status` got the DEFAULT value `'PENDING'`.
* Inserting an explicit value into `row_id` (GENERATED ALWAYS AS IDENTITY) **failed** with `[DELTA_IDENTITY_COLUMNS_EXPLICIT_INSERT_NOT_SUPPORTED] Providing values for GENERATED ALWAYS AS IDENTITY column row_id is not supported.`

> **Note:** `DEFAULT` values require `delta.feature.allowColumnDefaults = 'supported'` as a TBLPROPERTIES. Without it the CREATE TABLE fails with `[WRONG_COLUMN_DEFAULTS_FOR_DELTA_FEATURE_NOT_ENABLED]`.
>
> **Exam point:** `GENERATED ALWAYS` columns (both identity and computed) reject explicit inserts. `GENERATED BY DEFAULT AS IDENTITY` allows explicit values.

### Example E7 — MERGE WITH SCHEMA EVOLUTION

**Predict:**
1. The target has `(id INT, val STRING)`. The source has `(id INT, val STRING, new_col STRING)`. A regular MERGE — would it fail because of the extra `new_col`?
2. `MERGE WITH SCHEMA EVOLUTION INTO` — does it add `new_col` to the target? What happens to matched rows that were updated?

In [0]:
%python
# E7: MERGE WITH SCHEMA EVOLUTION adds new columns to the target
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex7_target")
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex7_source")
spark.sql("CREATE TABLE main.dbacademy_l02.ex7_target (id INT, val STRING)")
spark.sql("INSERT INTO main.dbacademy_l02.ex7_target VALUES (1,'a'),(2,'b')")
spark.sql("CREATE TABLE main.dbacademy_l02.ex7_source (id INT, val STRING, new_col STRING)")
spark.sql("INSERT INTO main.dbacademy_l02.ex7_source VALUES (2,'b2','x'),(3,'c','y')")
print("target schema before merge:", [(r["col_name"], r["data_type"]) for r in spark.sql("DESCRIBE main.dbacademy_l02.ex7_target").collect()])
spark.sql("""
MERGE WITH SCHEMA EVOLUTION INTO main.dbacademy_l02.ex7_target AS t
USING main.dbacademy_l02.ex7_source AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
print("target schema after merge:", [(r["col_name"], r["data_type"]) for r in spark.sql("DESCRIBE main.dbacademy_l02.ex7_target").collect()])
display(spark.sql("SELECT * FROM main.dbacademy_l02.ex7_target ORDER BY id"))

target schema before merge: [('id', 'int'), ('val', 'string')]
target schema after merge: [('id', 'int'), ('val', 'string'), ('new_col', 'string')]


id,val,new_col
1,a,null
2,b2,x
3,c,y


**Explained (real output from this run):**

* Target schema before merge: `[(id, int), (val, string)]`.
* After `MERGE WITH SCHEMA EVOLUTION`: schema became `[(id, int), (val, string), (new_col, string)]` — the `new_col` was added.
* Final rows:
  * `id=1`: `(1, 'a', NULL)` — unmatched by source, unchanged, `new_col` is NULL.
  * `id=2`: `(2, 'b2', 'x')` — matched and updated.
  * `id=3`: `(3, 'c', 'y')` — inserted.

> **Exam point:** `MERGE WITH SCHEMA EVOLUTION INTO` evolves the target schema (adds new columns from the source) during the MERGE. Without it, the MERGE would fail because of the extra `new_col`. Per the [type widening docs](https://docs.databricks.com/aws/en/tables/features/type-widening/), it can also widen column types if type widening is enabled.

### Example E8 — Managed table, DROP, and UNDROP

**Predict:**
1. What does `DESCRIBE EXTENDED` show for `Type` on a managed UC table?
2. After `DROP TABLE`, can you `SELECT` from it?
3. After `UNDROP TABLE`, are the data and history restored?
4. Can you create an external table with `LOCATION '/Volumes/...'` on this serverless workspace?

In [0]:
%python
# E8: managed table DROP/UNDROP; external table attempt
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex8_mgr")
spark.sql("CREATE TABLE main.dbacademy_l02.ex8_mgr (id INT, val STRING)")
spark.sql("INSERT INTO main.dbacademy_l02.ex8_mgr VALUES (1,'a'),(2,'b')")
ext = spark.sql("DESCRIBE EXTENDED main.dbacademy_l02.ex8_mgr").collect()
type_row = [r for r in ext if r["col_name"] == "Type"]
print("DESCRIBE EXTENDED Type:", type_row[0]["data_type"] if type_row else "not found")
# DROP
spark.sql("DROP TABLE main.dbacademy_l02.ex8_mgr")
print("after DROP, SELECT fails:")
try:
    spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.ex8_mgr").show()
except Exception as e:
    print("  ", str(e).splitlines()[0])
# UNDROP
spark.sql("UNDROP TABLE main.dbacademy_l02.ex8_mgr")
cnt = spark.sql("SELECT COUNT(*) AS c FROM main.dbacademy_l02.ex8_mgr").collect()[0]["c"]
print(f"after UNDROP, count: {cnt}")
display(spark.sql("DESCRIBE HISTORY main.dbacademy_l02.ex8_mgr").select("version","operation").orderBy("version"))
# external table attempt
print("external table CREATE TABLE ... LOCATION '/Volumes/...' (expect to fail):")
try:
    spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex8_ext")
    spark.sql("CREATE TABLE main.dbacademy_l02.ex8_ext (id INT, val STRING) LOCATION '/Volumes/main/dbacademy_l02/data/ex8_ext'")
    print("  succeeded")
except Exception as e:
    print("  failed:", str(e).splitlines()[0])

DESCRIBE EXTENDED Type: MANAGED
after DROP, SELECT fails:
   [TABLE_OR_VIEW_NOT_FOUND] The table or view `main`.`dbacademy_l02`.`ex8_mgr` cannot be found. Verify the spelling and correctness of the schema and catalog.
after UNDROP, count: 2


version,operation
0,CREATE TABLE
1,WRITE


external table CREATE TABLE ... LOCATION '/Volumes/...' (expect to fail):
  failed: [RequestId=5545f1e2-b5cf-4e40-bbec-6e9992606351 ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Missing cloud file system scheme


**Explained (real output from this run):**

* `DESCRIBE EXTENDED` showed `Type: MANAGED`.
* After `DROP TABLE`, `SELECT` **failed** with `[TABLE_OR_VIEW_NOT_FOUND] The table or view ... cannot be found.`
* After `UNDROP TABLE`, the count was **2** and history showed `(v0, CREATE TABLE), (v1, WRITE)` — data and history were restored.
* Creating an external table with `LOCATION '/Volumes/...'` **failed** with `[INVALID_PARAMETER_VALUE] Missing cloud file system scheme`.

> **NOT VERIFIED ON SERVERLESS — concept only:** external tables need an `s3://` (or `abfss://`/`gs://`) URI or a registered UC external location, not a `/Volumes/...` path. The behavior — `DROP` deletes metadata only, data files remain — is described in concept 4.5 and the [managed vs external docs](https://docs.databricks.com/aws/en/tables/external/).
>
> **Exam point:** `DROP` on a managed table deletes data; `UNDROP` recovers it within the retention window (verified here). `DROP` on an external table deletes metadata only.

### Example E9 — Shallow vs deep CLONE

**Predict:**
1. A `SHALLOW CLONE` copies only metadata (shares data files). What is the history operation name?
2. You `UPDATE` a row in the shallow clone. Does the source table change?
3. Do shallow and deep clones have the same `numFiles` and `sizeInBytes` right after cloning?

In [0]:
%python
# E9: SHALLOW vs DEEP CLONE; prove writes to clone don't affect source
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex9_shallow")
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex9_deep")
spark.sql("CREATE TABLE main.dbacademy_l02.ex9_shallow SHALLOW CLONE main.dbacademy_l02.encounters")
spark.sql("CREATE TABLE main.dbacademy_l02.ex9_deep DEEP CLONE main.dbacademy_l02.encounters")
sd = spark.sql("DESCRIBE DETAIL main.dbacademy_l02.ex9_shallow").collect()[0]
dd = spark.sql("DESCRIBE DETAIL main.dbacademy_l02.ex9_deep").collect()[0]
print(f"shallow: numFiles={sd['numFiles']}, sizeInBytes={sd['sizeInBytes']}")
print(f"deep:    numFiles={dd['numFiles']}, sizeInBytes={dd['sizeInBytes']}")
print("shallow history:", [(r["version"], r["operation"]) for r in spark.sql("DESCRIBE HISTORY main.dbacademy_l02.ex9_shallow").select("version","operation").collect()])
print("deep history:", [(r["version"], r["operation"]) for r in spark.sql("DESCRIBE HISTORY main.dbacademy_l02.ex9_deep").select("version","operation").collect()])
# UPDATE a row in the shallow clone — source must NOT change
spark.sql("UPDATE main.dbacademy_l02.ex9_shallow SET encounter_type = 'CLONED' WHERE encounter_id = 1")
src = spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.encounters WHERE encounter_type = 'CLONED'").collect()[0][0]
cln = spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.ex9_shallow WHERE encounter_type = 'CLONED'").collect()[0][0]
print(f"source CLONED count (want 0): {src};  shallow clone CLONED count (want 1): {cln}")

shallow: numFiles=1, sizeInBytes=3923
deep:    numFiles=1, sizeInBytes=3923
shallow history: [(0, 'CLONE')]
deep history: [(0, 'CLONE')]
source CLONED count (want 0): 0;  shallow clone CLONED count (want 1): 1


**Explained (real output from this run):**

* Both clones had `numFiles=1, sizeInBytes=3923` immediately after cloning (the source `encounters` table has 1 file; both clones reference it).
* History for both showed a single version: `(v0, CLONE)` — the operation name is **CLONE**.
* After `UPDATE` on the shallow clone: source `CLONED` count = **0** (unchanged), clone `CLONED` count = **1**. Writes to the clone do **not** affect the source.

> **Exam point:** a shallow clone shares the source's data files (copy-on-write); a deep clone copies them. Both are independent for writes. Use a shallow clone for cheap dev/test copies.

### Example E10 — OPTIMIZE (file compaction)

**Predict:**
1. You do 20 single-row `INSERT`s. How many data files are there?
2. After `OPTIMIZE`, how many files remain?
3. What do the `operationMetrics` show for `numRemovedFiles` and `numAddedFiles`?

In [0]:
%python
# E10: 20 small appends, then OPTIMIZE to compact
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex10_opt")
spark.sql("CREATE TABLE main.dbacademy_l02.ex10_opt (id INT, val STRING)")
for i in range(20):
    spark.sql(f"INSERT INTO main.dbacademy_l02.ex10_opt VALUES ({i}, 'val_{i}')")
db = spark.sql("DESCRIBE DETAIL main.dbacademy_l02.ex10_opt").collect()[0]
print(f"numFiles before OPTIMIZE: {db['numFiles']}")
spark.sql("OPTIMIZE main.dbacademy_l02.ex10_opt")
da = spark.sql("DESCRIBE DETAIL main.dbacademy_l02.ex10_opt").collect()[0]
print(f"numFiles after OPTIMIZE: {da['numFiles']}")
# show the OPTIMIZE metrics from history
hist = spark.sql("DESCRIBE HISTORY main.dbacademy_l02.ex10_opt").select("version","operation","operationMetrics").orderBy("version").collect()
for r in hist:
    if r["operation"] == "OPTIMIZE":
        print(f"OPTIMIZE v{r['version']} metrics: {r['operationMetrics']}")

numFiles before OPTIMIZE: 20
numFiles after OPTIMIZE: 1
OPTIMIZE v21 metrics: {'numRemovedFiles': '20', 'numRemovedBytes': '20489', 'p25FileSize': '1898', 'numDeletionVectorsRemoved': '0', 'minFileSize': '1898', 'p75FileSize': '1898', 'p50FileSize': '1898', 'numAddedBytes': '1898', 'numAddedFiles': '1', 'maxFileSize': '1898'}


**Explained (real output from this run):**

* `numFiles` before OPTIMIZE: **20** (one file per INSERT).
* `numFiles` after OPTIMIZE: **1** — all 20 small files were compacted into one.
* `OPTIMIZE` metrics (v21): `numRemovedFiles=20`, `numAddedFiles=1`, `numRemovedBytes=20489`, `numAddedBytes=1898`.

> **Exam point:** `OPTIMIZE` compacts small files into larger ones (default target 1 GB). `OPTIMIZE ... ZORDER BY (col)` co-locates data by column value; **liquid clustering** is the modern replacement (later lesson). Predictive optimization may auto-run OPTIMIZE and add `OPTIMIZE` versions to history you did not write.

### Example E11 — VACUUM and retention

**Predict:**
1. `VACUUM ... DRY RUN` on a freshly created table — does it list any files?
2. `VACUUM ... RETAIN 0 HOURS` — does it succeed?
3. Can you set `spark.databricks.delta.retentionDurationCheck.enabled = false` on this serverless compute?

In [0]:
%python
# E11: VACUUM DRY RUN, retention check, and the retentionDurationCheck config
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex11_vac")
spark.sql("CREATE TABLE main.dbacademy_l02.ex11_vac AS SELECT * FROM main.dbacademy_l02.encounters")
print("VACUUM DRY RUN (expect 0 rows — no files beyond retention):")
display(spark.sql("VACUUM main.dbacademy_l02.ex11_vac DRY RUN"))
print("VACUUM RETAIN 0 HOURS (expect fail):")
try:
    spark.sql("VACUUM main.dbacademy_l02.ex11_vac RETAIN 0 HOURS")
    print("  succeeded")
except Exception as e:
    print("  failed:", str(e).splitlines()[0])
print("set retentionDurationCheck.enabled=false (expect to fail on serverless):")
try:
    spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
    print("  config set")
except Exception as e:
    print("  failed:", str(e).splitlines()[0])

VACUUM DRY RUN (expect 0 rows — no files beyond retention):


path


VACUUM RETAIN 0 HOURS (expect fail):
  failed: [DELTA_VACUUM_RETENTION_PERIOD_TOO_SHORT] The specified VACUUM retention period is too low and may corrupt this Delta table if any writes are in progress.
set retentionDurationCheck.enabled=false (expect to fail on serverless):
  failed: [CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION] Configuration spark.databricks.delta.retentionDurationCheck.enabled is not available.  SQLSTATE: 42K0I


**Explained (real output from this run):**

* `VACUUM DRY RUN` returned **0 rows** — the table is fresh; no files are beyond the 7-day `delta.deletedFileRetentionDuration` window.
* `VACUUM ... RETAIN 0 HOURS` **failed** with `[DELTA_VACUUM_RETENTION_PERIOD_TOO_SHORT] The specified VACUUM retention period is too low and may corrupt this Delta table if any writes are in progress.`
* Setting `spark.databricks.delta.retentionDurationCheck.enabled = false` **failed** with `[CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION] Configuration spark.databricks.delta.retentionDurationCheck.enabled is not available.` — this config cannot be changed on serverless.

> **NOT VERIFIED ON SERVERLESS — concept only:** VACUUM actually removes old data files, which limits how far back you can time-travel. On this compute the retention check cannot be bypassed, so the actual file-removal effect of VACUUM is documented from the [VACUUM docs](https://docs.databricks.com/aws/en/delta/delta-utility/#-vacuum) only.

### Example E12 — Deletion vectors and REORG TABLE APPLY (PURGE)

**Predict:**
1. With deletion vectors enabled (default in UC), when you `DELETE` rows, are data files rewritten? What metric shows the deletion vector?
2. `REORG TABLE ... APPLY (PURGE)` — what does it do to the deletion vectors? What operation name appears in history?

In [0]:
%python
# E12: DELETE uses deletion vectors; REORG APPLY (PURGE) physically removes them
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.ex12_dv")
spark.sql("CREATE TABLE main.dbacademy_l02.ex12_dv AS SELECT * FROM main.dbacademy_l02.encounters WHERE encounter_id <= 50")
cntb = spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.ex12_dv").collect()[0][0]
print(f"rows before delete: {cntb}")
spark.sql("DELETE FROM main.dbacademy_l02.ex12_dv WHERE encounter_type = 'EMERGENCY'")
cnta = spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.ex12_dv").collect()[0][0]
print(f"rows after delete: {cnta}")
# show DELETE metrics
hist = spark.sql("DESCRIBE HISTORY main.dbacademy_l02.ex12_dv").select("version","operation","operationMetrics").orderBy("version").collect()
for r in hist:
    if r["operation"] == "DELETE":
        print(f"DELETE v{r['version']} metrics: {r['operationMetrics']}")
# REORG TABLE APPLY (PURGE)
spark.sql("REORG TABLE main.dbacademy_l02.ex12_dv APPLY (PURGE)")
print("REORG APPLY (PURGE) done")
hist2 = spark.sql("DESCRIBE HISTORY main.dbacademy_l02.ex12_dv").select("version","operation","operationMetrics").orderBy("version").collect()
for r in hist2:
    if r["operation"] == "OPTIMIZE" and r["version"] == 2:
        print(f"post-REORG v{r['version']} ({r['operation']}) metrics: {r['operationMetrics']}")

rows before delete: 50
rows after delete: 36
DELETE v1 metrics: {'numRemovedFiles': '0', 'numRemovedBytes': '0', 'numCopiedRows': '0', 'numDeletionVectorsAdded': '1', 'executionTimeMs': '1547', 'numDeletionVectorsUpdated': '0', 'numAddedFiles': '0', 'numDeletionVectorsRemoved': '0', 'numAddedChangeFiles': '0', 'numDeletedRows': '14', 'scanTimeMs': '1097', 'numAddedBytes': '0', 'rewriteTimeMs': '450'}
REORG APPLY (PURGE) done
post-REORG v2 (OPTIMIZE) metrics: {'numRemovedFiles': '1', 'numRemovedBytes': '2714', 'p25FileSize': '2607', 'numDeletionVectorsRemoved': '1', 'minFileSize': '2607', 'p75FileSize': '2607', 'p50FileSize': '2607', 'numAddedBytes': '2607', 'numAddedFiles': '1', 'maxFileSize': '2607'}


**Explained (real output from this run):**

* `DELETE` of `EMERGENCY` rows: **50** rows → **36** rows (14 deleted).
* `DELETE` metrics: `numRemovedFiles=0`, `numAddedFiles=0`, `numDeletionVectorsAdded=1`, `numDeletedRows=14` — no files were rewritten; the 14 deletions were recorded as a **deletion vector** on the existing file.
* `REORG TABLE ... APPLY (PURGE)` appeared in history as an **OPTIMIZE** operation (v2) with `numDeletionVectorsRemoved=1`, `numRemovedFiles=1`, `numAddedFiles=1` — the file with the deletion vector was rewritten to physically remove the deleted rows.

> **Exam point:** deletion vectors (default ON in UC) let `DELETE`/`UPDATE` mark rows without rewriting files — the metric `numDeletionVectorsAdded` confirms this. `REORG TABLE ... APPLY (PURGE)` physically rewrites the files to purge soft-deleted data (see [Deletion vectors docs](https://docs.databricks.com/aws/en/delta/deletion-vectors/)).

## 6 Exam Traps

| # | Trap (what the exam implies) | Reality (verified here or docs) |
| --- | --- | --- |
| 1 | `mergeSchema` and `overwriteSchema` do the same thing | **No.** `mergeSchema` adds new columns (old rows get NULL); `overwriteSchema` replaces the entire schema (overwrite mode only). |
| 2 | CHECK constraints are informational only | **No.** CHECK and NOT NULL are **enforced**. Only PK/FK/UNIQUE are informational only (concept 4.3, [docs](https://docs.databricks.com/aws/en/tables/constraints/)). |
| 3 | `RENAME COLUMN` works by default | **No.** Needs `delta.columnMapping.mode = 'name'`. Error: `[DELTA_UNSUPPORTED_RENAME_COLUMN.ENABLE_COLUMN_MAPPING]` (E4). |
| 4 | `DROP TABLE` on an external table deletes the data files | **No.** Only metadata is deleted; data files remain (concept 4.5). |
| 5 | `VACUUM` deletes old transaction-log entries | **No.** VACUUM removes **data files** no longer referenced. `delta.logRetentionDuration` (30 days) controls the log, not VACUUM. |
| 6 | A shallow clone copies the data files | **No.** It shares the source's files (metadata only, copy-on-write). E9: both clones had 1 file; writes to clone didn't affect source. |
| 7 | You can `VACUUM RETAIN 0 HOURS` to force removal | **No.** Blocked by retentionDurationCheck (`[DELTA_VACUUM_RETENTION_PERIOD_TOO_SHORT]`); config not available on serverless (E11). |
| 8 | `ALTER COLUMN ... TYPE` always works | **No.** Wider types need `delta.enableTypeWidening = true` (E3, [docs](https://docs.databricks.com/aws/en/tables/features/type-widening/)). |
| 9 | You can insert an explicit value into a GENERATED ALWAYS column | **No.** `[DELTA_IDENTITY_COLUMNS_EXPLICIT_INSERT_NOT_SUPPORTED]` (E6). |
| 10 | Adding a CHECK constraint on a table with violating rows just ignores them | **No.** The ALTER **fails** — you must clean the bad rows first (E5: `[DELTA_NEW_CHECK_CONSTRAINT_VIOLATION]`). |
| 11 | `DELETE` always rewrites the data files | **No.** With deletion vectors (default), rows are marked, not rewritten (`numDeletionVectorsAdded=1`, `numRemovedFiles=0`, E12). |
| 12 | `OPTIMIZE` and `VACUUM` do the same thing | **No.** OPTIMIZE compacts small files into larger ones; VACUUM removes unreferenced old files. |
| 13 | `DEFAULT` values work without any setup | **No.** Need `delta.feature.allowColumnDefaults = 'supported'` or the CREATE TABLE fails (E6). |
| 14 | `MERGE WITH SCHEMA EVOLUTION` requires the target to already have the new columns | **No.** It adds new columns during the MERGE (E7). |

## 7 Cheat Sheet — SQL ↔ PySpark for Delta operations

### Schema

| Operation | SQL | PySpark |
| --- | --- | --- |
| Add column | `ALTER TABLE t ADD COLUMN (col TYPE)` | `spark.sql("ALTER TABLE t ADD COLUMN (col TYPE)")` |
| Rename column | `ALTER TABLE t RENAME COLUMN old TO new` *(needs column mapping)* | same SQL via `spark.sql(...)` |
| Drop column | `ALTER TABLE t DROP COLUMN col` *(needs column mapping)* | same SQL via `spark.sql(...)` |
| Widen type | `ALTER TABLE t ALTER COLUMN col TYPE bigint` *(needs type widening)* | same SQL via `spark.sql(...)` |
| Merge schema on write | — | `.option("mergeSchema","true")` |
| Replace schema on overwrite | — | `.option("overwriteSchema","true")` |

### Constraints

| Operation | SQL | PySpark |
| --- | --- | --- |
| NOT NULL | `ALTER TABLE t ALTER COLUMN c SET NOT NULL` | same SQL via `spark.sql(...)` |
| CHECK | `ALTER TABLE t ADD CONSTRAINT chk CHECK (cond)` | same SQL via `spark.sql(...)` |
| Show constraints | `SHOW TBLPROPERTIES t` | `spark.sql("SHOW TBLPROPERTIES t").show()` |

### Clones

| Operation | SQL |
| --- | --- |
| Shallow clone | `CREATE TABLE dev SHALLOW CLONE source` |
| Deep clone | `CREATE TABLE backup DEEP CLONE source` |

### Maintenance

| Operation | SQL |
| --- | --- |
| Compact files | `OPTIMIZE t` |
| Co-locate by column | `OPTIMIZE t ZORDER BY (col)` |
| Dry-run vacuum | `VACUUM t DRY RUN` |
| Vacuum | `VACUUM t RETAIN 168 HOURS` |
| Purge deleted rows | `REORG TABLE t APPLY (PURGE)` |

### History / detail

| Operation | SQL |
| --- | --- |
| Audit trail | `DESCRIBE HISTORY t` |
| Physical info | `DESCRIBE DETAIL t` |
| Schema + metadata | `DESCRIBE EXTENDED t` |
| Time travel | `SELECT * FROM t VERSION AS OF 0` |
| Restore | `RESTORE TABLE t TO VERSION AS OF 0` |
| Undrop | `UNDROP TABLE t` |

### Generated / identity

| Feature | Syntax |
| --- | --- |
| Generated column | `col TYPE GENERATED ALWAYS AS (expr)` |
| Identity (no explicit) | `col BIGINT GENERATED ALWAYS AS IDENTITY` |
| Identity (explicit ok) | `col BIGINT GENERATED BY DEFAULT AS IDENTITY` |
| Default | `col TYPE DEFAULT 'value'` *(needs `allowColumnDefaults`)* |

### Key table properties

| Property | Effect |
| --- | --- |
| `delta.columnMapping.mode = 'name'` | enables RENAME/DROP COLUMN |
| `delta.enableTypeWidening = 'true'` | enables ALTER COLUMN TYPE to wider type |
| `delta.feature.allowColumnDefaults = 'supported'` | enables DEFAULT values |
| `delta.deletedFileRetentionDuration` | retention before VACUUM can remove (default 7 days) |
| `delta.logRetentionDuration` | how long the transaction log is kept (default 30 days) |
| `delta.checkpointInterval` | commits between checkpoints (default 10) |

## 8 Practice

Type these from memory. Each task has a validation cell. Solutions are in section 11.

### P1 — Add a CHECK constraint safely

Copy `patients` to `p1_patients`. Before adding, count rows that would violate `gender IN ('M','F')`. Then add the CHECK constraint and try inserting a row with `gender = 'X'` (should fail).

In [0]:
%python
# P1: copy patients, check for violations, add CHECK, test a violating insert
# TODO: your code here


In [0]:
%python
# P1 validation
try:
    props = spark.sql("SHOW TBLPROPERTIES main.dbacademy_l02.p1_patients").collect()
    has_check = any("constraints" in r["key"].lower() for r in props)
    if has_check:
        print("✅ Passed — P1 CHECK constraint added")
    else:
        print("Hint: no CHECK constraint found in TBLPROPERTIES")
except Exception as e:
    print("Hint: write and run the P1 code cell first.", str(e).splitlines()[0])

### P2 — Shallow clone, modify, verify source unchanged

Create a shallow clone `p2_lr_clone` of `lab_results`. Delete all rows with NULL `patient_id` from the clone. Verify: (a) the clone has fewer rows, (b) the source `lab_results` still has 13 NULL patient_ids.

In [0]:
%python
# P2: shallow clone lab_results, delete NULL patient_ids from clone, verify source unchanged
# TODO: your code here


In [0]:
%python
# P2 validation
try:
    src_nulls = spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.lab_results WHERE patient_id IS NULL").collect()[0][0]
    clone_nulls = spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.p2_lr_clone WHERE patient_id IS NULL").collect()[0][0]
    if src_nulls == 13 and clone_nulls == 0:
        print(f"✅ Passed — P2 clone (source NULLs={src_nulls}, clone NULLs={clone_nulls})")
    else:
        print(f"Hint: source NULLs={src_nulls} (want 13), clone NULLs={clone_nulls} (want 0)")
except Exception as e:
    print("Hint: write and run the P2 code cell first.", str(e).splitlines()[0])

### P3 — OPTIMIZE and verify file compaction

Create `p3_opt (id INT, val STRING)`, do 15 single-row INSERTs, record numFiles before OPTIMIZE, run OPTIMIZE, record numFiles after. Print both numbers.

In [0]:
%python
# P3: 15 small appends, OPTIMIZE, show before/after numFiles
# TODO: your code here


In [0]:
%python
# P3 validation
try:
    d = spark.sql("DESCRIBE DETAIL main.dbacademy_l02.p3_opt").collect()[0]
    if d["numFiles"] <= 2:
        print(f"✅ Passed — P3 OPTIMIZE (numFiles after = {d['numFiles']})")
    else:
        print(f"Hint: numFiles={d['numFiles']} (want <=2 after OPTIMIZE)")
except Exception as e:
    print("Hint: write and run the P3 code cell first.", str(e).splitlines()[0])

### P4 — Identity + generated columns

Create `p4_encounters` with `row_id BIGINT GENERATED ALWAYS AS IDENTITY`, `encounter_id INT`, `encounter_date DATE`, `encounter_year INT GENERATED ALWAYS AS (YEAR(encounter_date))`. Insert 3 rows with dates in 2023, 2024, 2025 (specifying only encounter_id and encounter_date). Verify row_id is 1, 2, 3 and encounter_year matches each date.

In [0]:
%python
# P4: identity + generated columns, insert 3 rows, verify
# TODO: your code here


In [0]:
%python
# P4 validation
try:
    rows = spark.sql("SELECT row_id, encounter_year FROM main.dbacademy_l02.p4_encounters ORDER BY row_id").collect()
    if len(rows) == 3 and rows[0]["row_id"] == 1 and rows[1]["row_id"] == 2 and rows[2]["row_id"] == 3:
        years = [r["encounter_year"] for r in rows]
        if years == [2023, 2024, 2025]:
            print(f"✅ Passed — P4 identity+generated (row_ids=1,2,3, years={years})")
        else:
            print(f"Hint: years={years} (want [2023,2024,2025])")
    else:
        print(f"Hint: row_ids={[r['row_id'] for r in rows]} (want [1,2,3])")
except Exception as e:
    print("Hint: write and run the P4 code cell first.", str(e).splitlines()[0])

## 9 Exam Questions (multiple choice — answers in section 11)

**Q1.** You append a DataFrame with a new column to a Delta table **without** `mergeSchema`. What happens?

a) The new column is added automatically  b) The write fails with `[DELTA_METADATA_MISMATCH]`  c) The new column is ignored  d) The table is recreated

**Q2.** Which constraints are **enforced** at write time in Unity Catalog?

a) PK and FK  b) CHECK and NOT NULL  c) UNIQUE and CHECK  d) All of the above

**Q3.** What table property must you enable before `RENAME COLUMN`?

a) `delta.enableTypeWidening`  b) `delta.columnMapping.mode = 'name'`  c) `delta.feature.allowColumnDefaults`  d) None — it works by default

**Q4.** A shallow clone shares the source's data files. After you `UPDATE` a row in the clone, what happens to the source?

a) The source row is also updated  b) The source is unchanged (copy-on-write)  c) The source is locked  d) The clone is detached

**Q5.** `VACUUM ... RETAIN 0 HOURS` on a serverless workspace fails with which error?

a) `[TABLE_OR_VIEW_NOT_FOUND]`  b) `[DELTA_VACUUM_RETENTION_PERIOD_TOO_SHORT]`  c) `[CONFIG_NOT_AVAILABLE]`  d) `[DELTA_METADATA_MISMATCH]`

**Q6.** Which command permanently changes a column from `INT` to `BIGINT` without rewriting data?

a) `ALTER TABLE t ALTER COLUMN c TYPE BIGINT` (with `delta.enableTypeWidening`)  b) `mergeSchema`  c) `overwriteSchema`  d) `DROP + CREATE`

**Q7.** With deletion vectors enabled, what does `DELETE` write to the table?

a) New data files with the remaining rows  b) A deletion vector marking the removed rows (no file rewrite)  c) A new transaction-log checkpoint  d) Nothing — DELETE is deferred

**Q8.** You add a CHECK constraint to a table that has rows violating it. What happens?

a) The constraint is added and bad rows are deleted  b) The constraint is added and bad rows are set to NULL  c) The ALTER fails with `[DELTA_NEW_CHECK_CONSTRAINT_VIOLATION]`  d) The constraint is added but not enforced

## 10 Cleanup

```python
# Uncomment to drop all lesson tables when you are done:
# spark.sql("DROP SCHEMA IF EXISTS main.dbacademy_l02 CASCADE")
# print("Schema main.dbacademy_l02 dropped")
```

Re-run the setup cell (section 3) to recreate everything.

## 11 Solutions

### Warm-up solutions

**W1 — MERGE with all three branches**
```python
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.wu1_target")
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.wu1_source")
spark.sql("CREATE TABLE main.dbacademy_l02.wu1_target (id INT, status STRING)")
spark.sql("INSERT INTO main.dbacademy_l02.wu1_target VALUES (1,'old'),(2,'old'),(3,'old')")
spark.sql("CREATE TABLE main.dbacademy_l02.wu1_source (id INT, status STRING)")
spark.sql("INSERT INTO main.dbacademy_l02.wu1_source VALUES (1,'new'),(4,'new')")
spark.sql("""
MERGE INTO main.dbacademy_l02.wu1_target AS t
USING main.dbacademy_l02.wu1_source AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.status = s.status
WHEN NOT MATCHED THEN INSERT (id, status) VALUES (s.id, s.status)
WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
```

**W2 — Time travel to the first version**
```python
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.wu2_t")
spark.sql("CREATE TABLE main.dbacademy_l02.wu2_t AS SELECT cast(id as INT) AS id FROM range(1,6)")  # 5 rows -> v0
spark.sql("INSERT INTO main.dbacademy_l02.wu2_t SELECT cast(id as INT) AS id FROM range(6,11)")     # 5 more -> v1
print(spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.wu2_t VERSION AS OF 0").collect()[0][0])  # 5
print(spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.wu2_t").collect()[0][0])                  # 10
```

**W3 — RESTORE with retry**
```python
import time
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.wu3_t")
spark.sql("CREATE TABLE main.dbacademy_l02.wu3_t (id INT)")
spark.sql("INSERT INTO main.dbacademy_l02.wu3_t VALUES " + ",".join([f"({i})" for i in range(1,11)]))
pre = spark.sql("DESCRIBE HISTORY main.dbacademy_l02.wu3_t").select("version").orderBy("version").collect()[-1]["version"]
spark.sql("DELETE FROM main.dbacademy_l02.wu3_t WHERE id <= 4")
for _ in range(6):
    try:
        spark.sql(f"RESTORE TABLE main.dbacademy_l02.wu3_t TO VERSION AS OF {pre}"); break
    except Exception as e:
        if "ConcurrentWrite" in str(e): time.sleep(3); continue
        raise
```

### Practice solutions

**P1 — Add a CHECK constraint safely**
```python
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.p1_patients")
spark.sql("CREATE TABLE main.dbacademy_l02.p1_patients AS SELECT * FROM main.dbacademy_l02.patients")
bad = spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.p1_patients WHERE gender NOT IN ('M','F')").collect()[0][0]
print("violating rows:", bad)  # 0
spark.sql("ALTER TABLE main.dbacademy_l02.p1_patients ADD CONSTRAINT chk_gender CHECK (gender IN ('M','F'))")
try:
    spark.sql("INSERT INTO main.dbacademy_l02.p1_patients (patient_id, patient_name, dob, gender, city, created_at) VALUES (999, 'Test', '2000-01-01', 'X', 'Boston', '2024-01-01')")
except Exception as e:
    print("insert failed:", str(e).splitlines()[0])
```

**P2 — Shallow clone, modify, verify source unchanged**
```python
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.p2_lr_clone")
spark.sql("CREATE TABLE main.dbacademy_l02.p2_lr_clone SHALLOW CLONE main.dbacademy_l02.lab_results")
spark.sql("DELETE FROM main.dbacademy_l02.p2_lr_clone WHERE patient_id IS NULL")
print("clone rows:", spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.p2_lr_clone").collect()[0][0])
print("source NULL patient_ids:", spark.sql("SELECT COUNT(*) FROM main.dbacademy_l02.lab_results WHERE patient_id IS NULL").collect()[0][0])  # 13
```

**P3 — OPTIMIZE and verify file compaction**
```python
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.p3_opt")
spark.sql("CREATE TABLE main.dbacademy_l02.p3_opt (id INT, val STRING)")
for i in range(15):
    spark.sql(f"INSERT INTO main.dbacademy_l02.p3_opt VALUES ({i}, 'val_{i}')")
print("before:", spark.sql("DESCRIBE DETAIL main.dbacademy_l02.p3_opt").collect()[0]["numFiles"])  # 15
spark.sql("OPTIMIZE main.dbacademy_l02.p3_opt")
print("after:", spark.sql("DESCRIBE DETAIL main.dbacademy_l02.p3_opt").collect()[0]["numFiles"])   # 1
```

**P4 — Identity + generated columns**
```python
spark.sql("DROP TABLE IF EXISTS main.dbacademy_l02.p4_encounters")
spark.sql("CREATE TABLE main.dbacademy_l02.p4_encounters (row_id BIGINT GENERATED ALWAYS AS IDENTITY, encounter_id INT, encounter_date DATE, encounter_year INT GENERATED ALWAYS AS (YEAR(encounter_date)))")
spark.sql("INSERT INTO main.dbacademy_l02.p4_encounters (encounter_id, encounter_date) VALUES (1,'2023-03-15'),(2,'2024-06-20'),(3,'2025-09-10')")
display(spark.sql("SELECT * FROM main.dbacademy_l02.p4_encounters ORDER BY row_id"))
```

### Exam answers

| Q | Answer |
| --- | --- |
| Q1 | **b** — `[DELTA_METADATA_MISMATCH]` |
| Q2 | **b** — CHECK and NOT NULL are enforced |
| Q3 | **b** — `delta.columnMapping.mode = 'name'` |
| Q4 | **b** — source unchanged (copy-on-write) |
| Q5 | **b** — `[DELTA_VACUUM_RETENTION_PERIOD_TOO_SHORT]` |
| Q6 | **a** — `ALTER COLUMN TYPE` with type widening |
| Q7 | **b** — deletion vector, no file rewrite |
| Q8 | **c** — `[DELTA_NEW_CHECK_CONSTRAINT_VIOLATION]` |

## 12 Final Summary

### What you learned (with real numbers from this run)

* **Delta transaction log** — every write creates a commit; `DESCRIBE HISTORY` shows versions; `DESCRIBE DETAIL` shows physical info. E1: 3 writes → 4 versions, `numFiles=3`, `sizeInBytes=3006`.
* **Schema enforcement** — extra columns rejected with `[DELTA_METADATA_MISMATCH]`. `mergeSchema` adds columns (old rows get NULL — 2 rows). `overwriteSchema` replaces schema. (E2)
* **Type widening** — `ALTER COLUMN TYPE` to wider type needs `delta.enableTypeWidening`. E3: `INT`→`BIGINT` succeeded. (E3)
* **Column mapping** — `ADD COLUMN` works; `RENAME`/`DROP` need `delta.columnMapping.mode = 'name'`. (E4)
* **Constraints** — CHECK and NOT NULL are enforced. Adding on a table with violations fails: E5 had **26 negative rows** → `[DELTA_NEW_CHECK_CONSTRAINT_VIOLATION]`, **13 NULL patient_ids** → `[DELTA_NEW_NOT_NULL_VIOLATION]`. PK/FK/UNIQUE are informational only.
* **Generated/identity columns** — `GENERATED ALWAYS` auto-computes (E6: `row_id=1`, `encounter_year=2024`, `status=PENDING`). Explicit insert into identity fails with `[DELTA_IDENTITY_COLUMNS_EXPLICIT_INSERT_NOT_SUPPORTED]`. `DEFAULT` needs `allowColumnDefaults`.
* **MERGE WITH SCHEMA EVOLUTION** — adds new columns to the target during the MERGE (E7: `new_col` added).
* **Managed/external/UNDROP** — managed `Type=MANAGED`; DROP → `[TABLE_OR_VIEW_NOT_FOUND]`; UNDROP restores data (count=2) + history. External table on `/Volumes/` fails with `[INVALID_PARAMETER_VALUE]`. (E8)
* **CLONE** — shallow shares files (cheap, copy-on-write); deep copies files. E9: both had 1 file; UPDATE on clone → source unchanged (0 CLONED rows in source).
* **OPTIMIZE** — compacts small files. E10: **20 → 1** file, `numRemovedFiles=20`, `numAddedFiles=1`.
* **VACUUM** — DRY RUN returned 0 rows (nothing past retention). `RETAIN 0 HOURS` → `[DELTA_VACUUM_RETENTION_PERIOD_TOO_SHORT]`. Config `retentionDurationCheck.enabled` → `[CONFIG_NOT_AVAILABLE]` on serverless. (E11)
* **Deletion vectors** — DELETE marks rows without rewriting (`numDeletionVectorsAdded=1`, `numRemovedFiles=0`). E12: 50 → 36 rows (14 deleted). `REORG APPLY (PURGE)` physically removes the deletion vector (`numDeletionVectorsRemoved=1`, appears as `OPTIMIZE` in history).

### NOT VERIFIED ON SERVERLESS — concept only

* `_delta_log` JSON/checkpoint file listing — `[DBFS_DISABLED]`
* External table creation — needs `s3://` URI or UC external location, not `/Volumes/`
* VACUUM actual file removal — retention check cannot be bypassed

### Key error classes verified

| Error class | Cause | Example |
| --- | --- | --- |
| `DELTA_METADATA_MISMATCH` | extra/missing column on append | E2 |
| `CAST_INVALID_INPUT` | un-castable value on append | E3 |
| `DELTA_UNSUPPORTED_RENAME_COLUMN.ENABLE_COLUMN_MAPPING` | RENAME without column mapping | E4 |
| `DELTA_NEW_CHECK_CONSTRAINT_VIOLATION` | ADD CHECK with existing violations | E5 |
| `DELTA_VIOLATE_CONSTRAINT_WITH_VALUES` | INSERT violating CHECK | E5 |
| `DELTA_NEW_NOT_NULL_VIOLATION` | SET NOT NULL with existing NULLs | E5 |
| `DELTA_IDENTITY_COLUMNS_EXPLICIT_INSERT_NOT_SUPPORTED` | explicit insert into identity | E6 |
| `TABLE_OR_VIEW_NOT_FOUND` | SELECT after DROP | E8 |
| `INVALID_PARAMETER_VALUE` | external table on `/Volumes/` path | E8 |
| `DELTA_VACUUM_RETENTION_PERIOD_TOO_SHORT` | VACUUM RETAIN 0 HOURS | E11 |
| `CONFIG_NOT_AVAILABLE` | retentionDurationCheck on serverless | E11 |
| `DBFS_DISABLED` | `dbutils.fs.ls` on managed table | E1 |